# Causal Forest — IACV

Análisis de efectos heterogéneos de tratamiento usando Causal Forest (`econml`). A diferencia de los modelos predictivos (Elastic Net, RF, XGBoost), este modelo estima **cómo varía el efecto de una intervención** según las características del municipio.

El resultado no es una predicción de violencia atípica, sino una estimación del **efecto causal heterogéneo** (CATE) de la variable de tratamiento sobre el outcome, condicional a las covariables.

**Requisito:** `pip install econml`

In [ ]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

_here = Path().resolve()
_root = _here
while not (_root / 'paths.yml').exists() and _root.parent != _root:
    _root = _root.parent

with open(_root / 'paths.yml') as f:
    _paths = yaml.safe_load(f)

processed = Path(_paths['data']['processed'])
model_out = Path(_paths['outputs']['model'])
fig_out   = Path(_paths['outputs']['figures']) / 'ct_iacv'
fig_out.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(_here))

from pipeline import load_data, create_temporal_splits, prepare_features, impute_missing
from econml.dml import CausalForestDML

print('pipeline y econml importados correctamente')

---
## CONFIGURACIÓN

**Variables del modelo causal:**
- `Y` (outcome): violencia atípica basada en IACV
- `T` (treatment): variable de intervención — el efecto que se quiere estimar
- `X` (heterogeneity): variables que modulan el efecto del tratamiento
- `W` (confounders): controles adicionales

In [ ]:
EXPERIMENT_NAME = 'cf_iacv'

DATA_PATH   = processed / 'db.parquet'
RESULTS_DIR = model_out

# ── Variables del modelo causal ────────────────────────────────────────────
OUTCOME_COL   = 'atypical_violence_iacv'   # Y: outcome binario
TREATMENT_COL = 'coca'                     # T: variable de tratamiento
TIME_COL      = 'quarter'
MUNICIPALITY_COL = 'mun_code'

# ── Covariables de heterogeneidad (X) ──────────────────────────────────────
# Variables que pueden modular el efecto de T sobre Y.
HETEROGENEITY_COLS = [
    'population', 'women_share',
    'nbi_2018', 'IPM_2018',
    'indrural', 'altura', 'discapital', 'disbogota',
    'areaoficialkm2', 'distancia_mercado',
    'DF_desemp_fisc', 'DF_ing_func',
]

# ── Controles / confounders (W) ────────────────────────────────────────────
# Variables que afectan tanto T como Y y deben controlarse.
CONFOUNDER_COLS = [
    'iacv_r1', 'iacv_r2', 'iacv_r3', 'iacv_r4',
    'dept_code',
    'petroleo_crudo_median', 'cafe_arabica_median', 'oro_median',
    'covid', 'covid_d',
    'docen_total', 'alumn_total',
    's11_total', 'DF_deuda',
]

# ── Splits ─────────────────────────────────────────────────────────────────
TRAIN_PROP = 0.70
VAL_PROP   = 0.15
TEST_PROP  = 0.15
USE_YEAR_SPLITS = False

# ── Causal Forest ──────────────────────────────────────────────────────────
N_ESTIMATORS = 500
MIN_SAMPLES_LEAF = 10
RANDOM_STATE = 42

# ── Imputación ─────────────────────────────────────────────────────────────
NUMERIC_STRATEGY     = 'median'
CATEGORICAL_STRATEGY = 'most_frequent'

print(f'configuración lista  |  experimento: {EXPERIMENT_NAME}')
print(f'  outcome   (Y): {OUTCOME_COL}')
print(f'  treatment (T): {TREATMENT_COL}')
print(f'  heterogeneity (X): {len(HETEROGENEITY_COLS)} variables')
print(f'  confounders   (W): {len(CONFOUNDER_COLS)} variables')

---
## Paso 1 — Cargar datos y splits

In [ ]:
df = load_data(DATA_PATH, time_col=TIME_COL, municipality_col=MUNICIPALITY_COL)

splits = create_temporal_splits(
    df, time_col=TIME_COL,
    train_prop=TRAIN_PROP, val_prop=VAL_PROP, test_prop=TEST_PROP,
    use_year_splits=USE_YEAR_SPLITS,
)

train_mask = splits['train_mask']
test_mask  = splits['test_mask']

---
## Paso 2 — Preparar variables del modelo causal

In [ ]:
# Outcome
Y = df[OUTCOME_COL].values

# Treatment
T = df[TREATMENT_COL].values

# Heterogeneity features
X = df[HETEROGENEITY_COLS].copy()

# Confounders
W = df[CONFOUNDER_COLS].copy()

# Imputar NaN con mediana
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)
W_imp = pd.DataFrame(imp.fit_transform(W), columns=W.columns, index=W.index)

# Splits
Y_train, Y_test = Y[train_mask], Y[test_mask]
T_train, T_test = T[train_mask], T[test_mask]
X_train, X_test = X_imp[train_mask].values, X_imp[test_mask].values
W_train, W_test = W_imp[train_mask].values, W_imp[test_mask].values

print(f'train: {train_mask.sum():,} obs | test: {test_mask.sum():,} obs')
print(f'prevalencia Y train: {Y_train.mean():.2%} | test: {Y_test.mean():.2%}')
print(f'media T train: {T_train.mean():.3f} | test: {T_test.mean():.3f}')

---
## Paso 3 — Ajustar Causal Forest

El `CausalForestDML` de `econml` estima efectos heterogéneos de tratamiento usando el método de double machine learning (DML): primero residualiza Y y T con respecto a W, luego estima el efecto causal sobre los residuos.

In [ ]:
cf = CausalForestDML(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

cf.fit(Y_train, T_train, X=X_train, W=W_train)
print('causal forest ajustado')

---
## Paso 4 — Estimar CATE en test

In [ ]:
# CATE: conditional average treatment effect
cate_test = cf.effect(X_test)

# Intervalos de confianza
cate_ci = cf.effect_interval(X_test, alpha=0.05)

print(f'CATE en test:')
print(f'  media  : {cate_test.mean():.4f}')
print(f'  mediana: {np.median(cate_test):.4f}')
print(f'  std    : {cate_test.std():.4f}')
print(f'  rango  : [{cate_test.min():.4f}, {cate_test.max():.4f}]')
print(f'  % positivos: {(cate_test > 0).mean():.2%}')

---
## Paso 5 — Distribución del efecto causal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma del CATE
axes[0].hist(cate_test, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].axvline(cate_test.mean(), color='orange', linestyle='-', linewidth=2, label=f'media: {cate_test.mean():.4f}')
axes[0].set_xlabel(f'CATE (efecto de {TREATMENT_COL})')
axes[0].set_ylabel('frecuencia')
axes[0].set_title('Distribución del efecto causal heterogéneo')
axes[0].legend()

# CATE vs tratamiento
axes[1].scatter(T_test, cate_test, alpha=0.1, s=5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel(f'{TREATMENT_COL}')
axes[1].set_ylabel('CATE')
axes[1].set_title(f'CATE vs {TREATMENT_COL}')

plt.tight_layout()
plt.savefig(fig_out / 'cate_distribucion.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Paso 6 — Importancia de features para heterogeneidad

Indica cuáles variables de `X` son las que más modulan el efecto del tratamiento.

In [ ]:
fi = cf.feature_importances_

df_fi = pd.DataFrame({
    'Feature': HETEROGENEITY_COLS,
    'Importance': fi,
}).sort_values('Importance', ascending=False)

print('Features que más modulan el efecto causal:')
print(df_fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = range(len(df_fi) - 1, -1, -1)
ax.barh(list(y_pos), df_fi['Importance'].values, color='steelblue')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(df_fi['Feature'].values)
ax.set_xlabel('Importancia para heterogeneidad del efecto')
ax.set_title(f'Drivers de heterogeneidad en el efecto de {TREATMENT_COL}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(fig_out / 'feature_importance_heterogeneidad.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Paso 7 — Análisis por subgrupos

In [ ]:
# Construir DataFrame de resultados en test
df_test = df[test_mask].copy()
df_test['cate'] = cate_test
df_test['cate_lower'] = cate_ci[0]
df_test['cate_upper'] = cate_ci[1]
df_test['efecto_significativo'] = (
    (df_test['cate_lower'] > 0) | (df_test['cate_upper'] < 0)
).astype(int)

print(f'observaciones con efecto significativo (IC 95% no cruza cero): '
      f'{df_test["efecto_significativo"].mean():.2%}')

# CATE promedio por quintil de ruralidad
df_test['quintil_rural'] = pd.qcut(df_test['indrural'], q=5, labels=['Q1 (urbano)', 'Q2', 'Q3', 'Q4', 'Q5 (rural)'])

cate_rural = df_test.groupby('quintil_rural')['cate'].agg(['mean', 'std', 'count'])
print(f'\nCATE promedio por quintil de ruralidad:')
print(cate_rural.to_string())

---
## Paso 8 — Exportar resultados

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# CATE por municipio-trimestre
cate_path = RESULTS_DIR / f'{EXPERIMENT_NAME}_cate_{timestamp}.csv'
df_test[['mun_code', 'quarter', 'cate', 'cate_lower', 'cate_upper', 'efecto_significativo']].to_csv(
    cate_path, index=False
)
print(f'CATE exportado: {cate_path}')

# Feature importance
fi_path = RESULTS_DIR / f'{EXPERIMENT_NAME}_feature_importance_{timestamp}.csv'
df_fi.to_csv(fi_path, index=False)
print(f'Feature importance exportado: {fi_path}')

# Resumen
resumen = {
    'experiment': EXPERIMENT_NAME,
    'outcome': OUTCOME_COL,
    'treatment': TREATMENT_COL,
    'n_heterogeneity_features': len(HETEROGENEITY_COLS),
    'n_confounder_features': len(CONFOUNDER_COLS),
    'n_estimators': N_ESTIMATORS,
    'train_size': int(train_mask.sum()),
    'test_size': int(test_mask.sum()),
    'cate_mean': float(cate_test.mean()),
    'cate_median': float(np.median(cate_test)),
    'cate_std': float(cate_test.std()),
    'pct_efecto_positivo': float((cate_test > 0).mean()),
    'pct_efecto_significativo': float(df_test['efecto_significativo'].mean()),
}
summary_path = RESULTS_DIR / f'{EXPERIMENT_NAME}_summary_{timestamp}.csv'
pd.DataFrame([resumen]).to_csv(summary_path, index=False)
print(f'Resumen exportado: {summary_path}')